# TP — Détection de beaconing C2

On dispose d'un dataframe contenant les logs de connexion sortantes sur un réseau d'entreprise.

Certains hôtes sont compromis, et font des appels réseau à un serveur « Commande et contrôle » (C2). Le but du TD est de
- identifier les hôtes compromis
- extraire les informations de compromission


## 0. Chargement des données

Le fichier `logs_reseau.csv` contient les connexions sortantes observées pendant
12 h (08:00 → 20:00) sur un réseau d'entreprise.

| colonne | description |
| --- | --- |
| `timestamp` | instant de la connexion |
| `src_ip` | hôte source (interne) |
| `dst_ip` | IP de destination |
| `domain` | domaine contacté (nom DNS) |
| `dst_port` | port de destination |
| `bytes_out` | octets envoyés |

> [!info] Aide pandas
> - `pd.read_csv(..., parse_dates=["timestamp"])` : charge et convertit la colonne en dates.
> - `.head()`, `.info()`, `.describe()` : premier aperçu.
> - Doc : https://pandas.pydata.org/docs/user_guide/10min.html


In [ ]:
import pandas as pd

df_logs_reseau = pd.read_csv("logs_reseau.csv", parse_dates=["timestamp"])
print(df_logs_reseau.shape)
df_logs_reseau.head()


In [ ]:
df_logs_reseau.info()


## Exercice 1 — Beaconing « classique »

Un implant « beacon » rappelle son serveur C2 à intervalle **régulier**. Objectif :
repérer le couple (hôte, domaine) au rythme le plus régulier.

Pour cela, on va calculer le « coefficient de variation » (CV), sur les intervalle de temps entre requête.

   - `CV ≈ 0` → intervalle quasi constant (beacon *ou* tâche planifiée) ;
   - `CV` élevé → navigation humaine, irrégulière.

### Calcul du coefficient de variation


Étant donnée une liste **triée** de timestamp de requêtes, on peut calculer le coefficient de variation de la manière suivante:
- on commence par créer une nouvelle liste contenant les intervalles de temps, **en secondes**, entre chaque couple de requête successifs.
- sur cette liste, on applique la formule: `CV = écart-type / moyenne`

**Méthode, pas à pas :**

1. **Regrouper en listes** : fabriquer un DataFrame où chaque ligne est un couple
   `(src_ip, domain)` et où l'on range ses timestamps dans une **liste** (triée).
2. **Transformer chaque liste en écarts** : une petite fonction Python
   (`get_diffs`) qui renvoie les temps inter-arrivées, en secondes.
3. **Résumer en CV** : Calculer le coefficient de variation pour chacune de ces listes

In [ ]:
def timestamps_par_src_dest(df: pd.DataFrame) -> pd.DataFrame:
    """
    À partir du dataframe logs_réseau, retourner un dataframe qui a en index
    (src_ip, domain), et une colonne `timestamps`, qui contient une liste triée 
    de tous les timestamps de connection associés à (src_ip, domain)

    Indice: on peut commencer par df.groupby(["src_ip", "domain"])
    """
    # TODO
    pass


In [ ]:
ts_per_src_dest = timestamps_par_src_dest(df_logs_reseau)
nb_couples = df_logs_reseau[["src_ip", "domain"]].drop_duplicates().shape[0]
assert ts_per_src_dest.shape[0] == nb_couples
assert "timestamps" in ts_per_src_dest.columns
assert ts_per_src_dest["timestamps"].map(lambda ts: ts == sorted(ts)).all()
print("Étape 1 OK :", nb_couples, "couples (src_ip, domain) distincts")


In [ ]:
from datetime import datetime


def get_diffs(timestamps: list[datetime]) -> list[float]:
    """Temps (en secondes) entre instants successifs, dans l'ordre chronologique.

    La liste est triée au besoin : aucune hypothèse sur l'ordre en entrée.
    """
    # TODO
    pass


In [ ]:
from datetime import datetime, timedelta

t0 = datetime(2026, 10, 14, 8, 0, 0)
assert get_diffs([t0, t0 + timedelta(seconds=60), t0 + timedelta(seconds=120)]) == [60.0, 60.0]
assert get_diffs([t0 + timedelta(seconds=120), t0, t0 + timedelta(seconds=60)]) == [60.0, 60.0]
assert get_diffs([t0]) == []
assert get_diffs([]) == []
print("get_diffs OK")


In [ ]:
import statistics


def cv_destinations(df: pd.DataFrame, n_min: int = 20) -> pd.DataFrame:
    """CV des temps inter-arrivées par couple (src_ip, domain).

    Colonnes retournées : src_ip, domain, n, cv. On ignore les couples trop
    rares (moins de n_min connexions) : un CV sur 2 points ne veut rien dire.
    """
    # TODO
    Le module `statistics` fournit `statistics.mean(liste)` et
    `statistics.stdev(liste)` : moyenne et écart-type d'une liste de nombres.


In [ ]:
cv_destinations(ts_per_src_dest).sort_values("cv")

In [ ]:
def detecter_beacon_timing(df: pd.DataFrame) -> tuple[str, str]:
    """Renvoie (src_ip, domain) du beacon le plus régulier, hors liste blanche."""
    # TODO
    pass


In [ ]:
src, domaine = detecter_beacon_timing(df_logs_reseau)
assert (src, domaine) == ("10.0.0.66", "sync-updates.example.net"), f"trouvé : {src} -> {domaine}"
print("ex 1 OK :", src, "->", domaine)

# Bonus : le beacon envoie toujours la même taille de payload (512 octets).
payloads = df_logs_reseau[df_logs_reseau["domain"] == "sync-updates.example.net"]["bytes_out"]
assert payloads.nunique() == 1
print("  -> payload constant :", payloads.iloc[0], "octets")


## Exercice 2 — Fast-flux : la destination qui tourne

Certains C2 changent d'IP **et** de domaine à chaque connexion pour échapper aux
filtres : c'est le **fast-flux** (ou DGA). Chaque domaine n'apparaissant qu'une
fois, grouper par domaine est inutile.

**Méthode :** regarder l'hôte *source*. Un hôte légitime contacte un petit nombre
de destinations. Un hôte en fast-flux contacte un grand nombre d'IP **distinctes**.


**Consignes :**

1. Créer un dataframe qui associe à chaque IP source le nombre d'IP de destination
   **distinctes**, puis l'afficher.
2. Écrire une fonction qui extrait l'IP source ayant le plus de destinations.
3. Écrire une fonction qui extrait les **IOC** (indicateurs de compromission)
   associés à cette IP.


In [ ]:
# TODO
pass


In [ ]:
def detecter_fastflux(df: pd.DataFrame) -> str:
    """Renvoie l'hôte source qui contacte le plus d'IP de destination distinctes."""
    # TODO
    pass


In [ ]:
def extraire_ioc(df: pd.DataFrame, src_ip: str) -> pd.DataFrame:
    """IOC associés à une IP source : les destinations (IP + domaine) contactées.

    Renvoie un dataframe des destinations distinctes (une ligne par IP de
    destination), trié par dst_ip.
    """
    # TODO
    pass


In [ ]:
hote = detecter_fastflux(df_logs_reseau)
assert hote == "10.0.0.67", f"trouvé : {hote}"
print("ex 2 OK :", hote)

ioc = extraire_ioc(df_logs_reseau, hote)
assert len(ioc) == 40
assert ioc["dst_ip"].is_unique
print("  ->", len(ioc), "IOC (IP de destination distinctes)")


## Exercice 3 — Timing aléatoire : il ne reste que la plage d'IP

L'implant le plus sophistiqué **randomise totalement** son timing : plus aucun
rythme à détecter. Mais il doit toujours joindre son serveur C2, hébergé dans un
`/24` que l'entreprise ne contacte jamais.

**Méthode :**

1. Réduire chaque IP à son préfixe `/24`.
2. Pour chaque préfixe, calculer **deux chiffres séparés** :
   - `nb_sources` : combien d'hôtes distincts le touchent ;
   - `volume` : combien de connexions il reçoit.
3. Un préfixe **rare** (1 seule source) **et massivement contacté** = suspect.

**Note** Le timing aléatoire ne laisse aucun rythme exploitable  
Pas de CV faible (timing aléatoire), peu d'IP distinctes : ni la régularité
 ni le comptage de destinations ne le détectent. Seule la plage d'IP le trahit.


In [ ]:
def prefixe(ip: str) -> str:
    """Préfixe /24 d'une IPv4."""
    # TODO
    pass


In [ ]:
def hotes_plage_rare(df: pd.DataFrame, vol_min: int = 50) -> list[str]:
    """Hôtes touchant un /24 rare (1 source) ET massivement contacté."""
    # TODO
    pass


In [ ]:
def cv_hote(df: pd.DataFrame, hote: str) -> float:
    """CV des temps inter-arrivées de toutes les connexions d'un hôte.

    Proche de 0 si l'hôte est régulier, élevé si son trafic est aléatoire.
    """
    # TODO
    pass


In [ ]:
suspects = hotes_plage_rare(df_logs_reseau)
assert suspects == ["10.0.0.66", "10.0.0.68"], f"trouvé : {suspects}"
print("ex 3 OK : préfixes rares touchés par", suspects)

# Des deux suspects, 10.0.0.66 contacte son C2 à intervalle régulier (CV faible).
# 10.0.0.68 n'a, lui, aucune périodicité (CV élevé) : c'est la cible.
assert cv_hote(df_logs_reseau, "10.0.0.68") > cv_hote(df_logs_reseau, "10.0.0.66")
print("  -> 10.0.0.68 n'a aucune périodicité (CV =", round(cv_hote(df_logs_reseau, "10.0.0.68"), 2), ")")


## Exercice 4 — Quand le comptage ne suffit plus

Le réseau grossit : on observe désormais une centaine d'hôtes, qui contactent
chacun des centaines d'IP de destination distinctes (CDN, edge, caches…).

Un hôte compromis renouvelle ses destinations à chaque connexion. Retrouvez-le.


In [ ]:
df_logs_reseau_grand = pd.read_csv("logs_reseau_grand.csv", parse_dates=["timestamp"])
print(df_logs_reseau_grand.shape)
df_logs_reseau_grand.head()


On vérifie d'abord que les méthodes des exercices 1 et 2 ne retrouvent plus
l'hôte compromis sur ce réseau.


In [ ]:
hote_fastflux = detecter_fastflux(df_logs_reseau_grand)
src_beacon, _ = detecter_beacon_timing(df_logs_reseau_grand)
assert hote_fastflux != "10.0.0.167", f"ex 2 désigne {hote_fastflux} au lieu de la victime"
assert src_beacon != "10.0.0.167", f"ex 1 désigne {src_beacon} au lieu de la victime"
print("méthode ex 2 →", hote_fastflux)
print("méthode ex 1 →", src_beacon)


**Consigne :** compléter `detecter_hote_compromis`, qui renvoie l'hôte compromis
du grand réseau.


In [ ]:
def detecter_hote_compromis(df: pd.DataFrame) -> str:
    """Renvoie l'hôte compromis du grand réseau."""
    # TODO
    pass


In [ ]:
victime = detecter_hote_compromis(df_logs_reseau_grand)
assert victime == "10.0.0.167", f"trouvé : {victime}"
print("ex 4 OK :", victime)


## Bilan

| exercice | signal | notion pandas | hôte trouvé |
| --- | --- | --- | --- |
| 1 | régularité temporelle | `groupby` + `agg(list)` + CV | `10.0.0.66` |
| 2 | multiplicité des destinations | `groupby().nunique()` | `10.0.0.67` |
| 3 | plage d'IP rare et concentrée | `map` + deux `groupby` | `10.0.0.68` |
| 4 | destinations rares (une seule source) | `nunique` + `isin` | `10.0.0.167` |

**À retenir :** la détection de C2 est **heuristique** — on croise plusieurs
signaux (rythme, destinations, plage d'IP, rareté des destinations). Aucun
signal seul ne suffit, et chaque contre-mesure de l'attaquant (jitter,
fast-flux, timing aléatoire, destinations partagées) casse un signal… mais en
laisse un autre.
